# Federated Learning Benchmark - Comparative Analysis
## NECSTLab - Polimi LS2

Confronto tra Flower Bagging, Flower Cyclic, e NVIDIA FLARE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Carica Risultati Benchmark

In [ ]:
# Carica risultati da CSV
results_dir = Path('../results')

# Lista tutti i file risultati
result_files = list(results_dir.glob('*.csv'))
print(f"Trovati {len(result_files)} file risultati")

# Carica e concatena
dfs = []
for f in result_files:
    df = pd.read_csv(f)
    dfs.append(df)

if dfs:
    results = pd.concat(dfs, ignore_index=True)
    print(f"\nCaricati {len(results)} esperimenti")
    display(results.head())
else:
    print("⚠️ Nessun risultato trovato. Esegui prima i benchmark!")

## 2. Confronto Tempo di Esecuzione

In [ ]:
if 'results' in locals():
    # Bar plot tempo totale
    fig, ax = plt.subplots(figsize=(10, 6))
    
    approaches = results.groupby('approach')['total_time_sec'].mean().sort_values()
    approaches.plot(kind='barh', ax=ax, color='skyblue')
    
    ax.set_xlabel('Tempo (secondi)', fontsize=12)
    ax.set_ylabel('Approccio', fontsize=12)
    ax.set_title('Confronto Tempo di Training', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Aggiungi valori sulle barre
    for i, v in enumerate(approaches):
        ax.text(v + 0.5, i, f'{v:.1f}s', va='center')
    
    plt.tight_layout()
    plt.savefig('../results/plots/time_comparison.png', dpi=300)
    plt.show()
    
    print(f"\n⚡ Approccio più veloce: {approaches.index[0]} ({approaches.iloc[0]:.1f}s)")
    print(f"🐌 Approccio più lento: {approaches.index[-1]} ({approaches.iloc[-1]:.1f}s)")
    print(f"📊 Speedup: {approaches.iloc[-1] / approaches.iloc[0]:.2f}x")

## 3. Confronto Accuratezza (MAE)

In [ ]:
if 'results' in locals() and 'final_mae' in results.columns:
    # Bar plot MAE
    fig, ax = plt.subplots(figsize=(10, 6))
    
    mae_by_approach = results.groupby('approach')['final_mae'].mean().sort_values()
    mae_by_approach.plot(kind='barh', ax=ax, color='coral')
    
    ax.set_xlabel('MAE (Mean Absolute Error)', fontsize=12)
    ax.set_ylabel('Approccio', fontsize=12)
    ax.set_title('Confronto Accuratezza', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for i, v in enumerate(mae_by_approach):
        ax.text(v + 0.2, i, f'{v:.2f}', va='center')
    
    plt.tight_layout()
    plt.savefig('../results/plots/accuracy_comparison.png', dpi=300)
    plt.show()
    
    print(f"\n🎯 Miglior accuratezza: {mae_by_approach.index[0]} (MAE: {mae_by_approach.iloc[0]:.2f})")
    print(f"📉 Peggior accuratezza: {mae_by_approach.index[-1]} (MAE: {mae_by_approach.iloc[-1]:.2f})")

## 4. Trade-off Accuracy vs Speed

In [ ]:
if 'results' in locals() and 'final_mae' in results.columns:
    # Scatter plot accuracy vs time
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for approach in results['approach'].unique():
        subset = results[results['approach'] == approach]
        ax.scatter(
            subset['total_time_sec'], 
            subset['final_mae'],
            s=200,
            alpha=0.7,
            label=approach
        )
    
    ax.set_xlabel('Tempo (secondi)', fontsize=12)
    ax.set_ylabel('MAE (lower is better)', fontsize=12)
    ax.set_title('Trade-off: Accuratezza vs Velocità', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    
    # Zona "sweet spot" (ipotetica)
    ax.axhspan(10, 12, alpha=0.1, color='green', label='Target accuracy zone')
    
    plt.tight_layout()
    plt.savefig('../results/plots/tradeoff_accuracy_speed.png', dpi=300)
    plt.show()

## 5. Analisi Dettagliata Timing

In [ ]:
if 'results' in locals():
    # Breakdown timing (se disponibile)
    timing_cols = [
        col for col in results.columns 
        if 'time' in col and col != 'timestamp'
    ]
    
    if len(timing_cols) > 1:
        timing_breakdown = results.groupby('approach')[timing_cols].mean()
        
        fig, ax = plt.subplots(figsize=(12, 6))
        timing_breakdown.plot(kind='bar', stacked=True, ax=ax)
        
        ax.set_ylabel('Tempo (secondi)', fontsize=12)
        ax.set_xlabel('Approccio', fontsize=12)
        ax.set_title('Breakdown Timing per Fase', fontsize=14, fontweight='bold')
        ax.legend(title='Fase', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('../results/plots/timing_breakdown.png', dpi=300)
        plt.show()
        
        print("\n📊 Timing Breakdown:")
        display(timing_breakdown)

## 6. Tabella Riassuntiva

In [ ]:
if 'results' in locals():
    summary = results.groupby('approach').agg({
        'total_time_sec': ['mean', 'std'],
        'final_mae': ['mean', 'std'],
        'num_clients': 'first',
        'num_rounds': 'first',
    }).round(2)
    
    print("\n" + "="*80)
    print("📊 SUMMARY BENCHMARK RESULTS")
    print("="*80)
    display(summary)
    
    # Salva summary
    summary.to_csv('../results/benchmark_summary.csv')
    print("\n✅ Summary salvato in results/benchmark_summary.csv")

## 7. Conclusioni e Next Steps

### Domande da Rispondere:

1. **Qual è il gap accuratezza tra Flower e FLARE?**
   - Se piccolo (<5% MAE) → Flower sufficiente
   - Se grande (>10% MAE) → Vale la pena cercare via di mezzo

2. **Dove si perde il tempo in FLARE?**
   - Communication overhead?
   - Histogram computation?
   - Aggregation?

3. **Quale approccio ibrido testare per primo?**
   - Se communication è il bottleneck → Compressed histograms
   - Se computation è il bottleneck → Selective histograms
   - Se convergenza è lenta → Adaptive communication

### Prossimi Esperimenti:

```python
# TODO: Implementare
- [ ] Flower Cyclic baseline
- [ ] NVIDIA FLARE baseline  
- [ ] Hybrid approach #1: Selective histograms
- [ ] Hybrid approach #2: Adaptive communication
- [ ] Confronto finale e paper
```